# Lab 4 - Memory: what an assistant remembers about *you*

Lab 3 stored conversations. But ChatGPT also remembers things **across** conversations -
"you're vegetarian", "you like short answers". That is **memory**, and there is more
than one kind:

| Memory type | What the name means | Injected into the prompt... | Where it lives |
|-------------|--------------------|-----------------------------|----------------|
| **Short-term** (conversation buffer) | only this chat, gone when it ends | ...every turn, whole transcript | the `messages` list you send the model (Lab 3) |
| **Long-term / persistent** | kept on disk across *every* chat until you delete it | depends - see the next two rows | a `memories` table on disk |
| &nbsp;&nbsp;• preferences | a long-term row about *how* you like answers | ...**always**, any topic | same table, `kind='preference'` |
| &nbsp;&nbsp;• **semantic recall** of facts/projects | a long-term row, fetched *by meaning* | ...**only when relevant** to the question | same table + an embedding per row |

"Long-term" and "semantic recall" are **not two stores** - one `memories` table. "Long-term"
is the *lifetime*; "semantic recall" is *how a row is chosen* for a given turn (embed the
message, compare to every memory, use the close ones).

This lab builds it properly:
- after each of your messages the **model reconciles** memory - it can **add**, **update**,
  or **delete** entries (you will watch it happen),
- your stated **preferences are always applied**; **facts/projects are recalled only when
  relevant** to the question,
- and **you can delete any memory** yourself - nothing is kept against your wishes.

Running scenario: someone asking a kitchen assistant what to cook.


## Step 0 - Install

In [ ]:
%pip install -q langchain langchain-groq langchain-openai langchain-huggingface \
    sentence-transformers numpy gradio

## Step 1 - Model: Groq first, OpenRouter as a fallback

Same wrapper as Lab 3. Primary is **Groq** `openai/gpt-oss-120b` (fast, and strong at the
JSON "reconcile" step). If Groq is down we fall through to a free **OpenRouter** model.
`.invoke()` returns the whole reply; `.stream()` yields it token by token (used by the
Gradio app in Step 12).

In [ ]:
import os

def load_key(name, required=True):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"{name}: from Colab secret"); return v
    except Exception:
        pass
    if os.getenv(name):
        print(f"{name}: from environment"); return os.environ[name]
    from getpass import getpass
    tail = "" if required else "  (optional - press Enter to skip)"
    return getpass(f"Paste {name}{tail}: ").strip()

GROQ_API_KEY       = load_key("GROQ_API_KEY")
OPENROUTER_API_KEY = load_key("OPENROUTER_API_KEY", required=False)
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

import time
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

MODELS = [ChatGroq(model="openai/gpt-oss-120b", api_key=GROQ_API_KEY,
                   temperature=0, max_retries=2, request_timeout=40)]
if OPENROUTER_API_KEY:
    for _m in ["nvidia/nemotron-3-super-120b-a12b:free", "minimax/minimax-m2.7:free"]:
        MODELS.append(ChatOpenAI(model=_m, base_url="https://openrouter.ai/api/v1",
                                 api_key=OPENROUTER_API_KEY, temperature=0,
                                 max_retries=1, request_timeout=40))

def _name(m):
    return getattr(m, "model_name", getattr(m, "model", "?"))

class LLM:
    """Try each backend in order; if all fail (e.g. rate limit) wait and retry the list.
    .invoke() -> full text; .stream() -> text chunks."""
    def invoke(self, messages):
        err = None
        for wait in (0, 12, 30):
            if wait:
                time.sleep(wait)
            for m in MODELS:
                try:
                    return m.invoke(messages).content
                except Exception as e:
                    err = e; print(f"  ({_name(m)} failed: {str(e)[:70]})")
        raise err

    def stream(self, messages):
        err = None
        for wait in (0, 12, 30):
            if wait:
                time.sleep(wait)
            for m in MODELS:
                try:
                    any_chunk = False
                    for chunk in m.stream(messages):
                        any_chunk = True
                        if chunk.content:
                            yield chunk.content
                    if any_chunk:
                        return
                except Exception as e:
                    err = e; print(f"  ({_name(m)} stream failed: {str(e)[:70]})")
        raise err

llm = LLM()
print("models:", [_name(m) for m in MODELS])
print(llm.invoke("Reply with one word: ready"))

## Step 2 - The `memories` table

One row per remembered thing, tagged with a `kind`:

- **preference** - how you like answers ("short numbered steps", "no long intros")
- **fact** - stable facts about you ("vegetarian", "allergic to peanuts")
- **project** - what you are currently working on ("meal-prepping lunches this week")


In [ ]:
import sqlite3, datetime, numpy as np

DB_PATH = "memory.db"
KINDS = ["preference", "fact", "project"]

def connect():
    return sqlite3.connect(DB_PATH)

with connect() as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS memories (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            kind       TEXT NOT NULL,
            content    TEXT NOT NULL,
            embedding  BLOB,               -- the row's vector, computed once at write time
            created_at TEXT NOT NULL
        )
    """)
print("memory.db ready")

## Step 3 - Embeddings + the memory operations

Every memory is embedded **once, when it is written**, and the vector is stored in the
`embedding` column. Recall and dedup then just read those vectors back - no re-embedding
the whole table on every turn (that was the slow part). Only *new* text (your latest
message, or a new memory) ever gets embedded.

The three operations the model can perform are plain SQL: `add`, `update`, `delete`.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

def _vec(text):
    return np.asarray(emb.embed_query(text), dtype="float32")   # 384-dim, unit length

def _stamp():
    return datetime.datetime.now().isoformat(timespec="seconds")

def all_memories():
    with connect() as con:
        return con.execute("SELECT id, kind, content FROM memories ORDER BY id").fetchall()

def all_memories_vec():
    """(id, kind, content, vector) - vector read straight from the stored BLOB."""
    with connect() as con:
        rows = con.execute("SELECT id, kind, content, embedding FROM memories ORDER BY id").fetchall()
    return [(i, k, c, np.frombuffer(b, dtype="float32")) for i, k, c, b in rows]

def add_memory(kind, content, verbose=True):
    kind = kind if kind in KINDS else "fact"
    cv = _vec(content)                                     # embed the NEW memory once
    for rid, rk, rc, rv in all_memories_vec():
        if float(cv @ rv) > 0.90:                          # near-identical -> already stored
            if verbose:
                print(f"  = already known: {content}")
            return rid
    with connect() as con:
        cur = con.execute(
            "INSERT INTO memories(kind, content, embedding, created_at) VALUES (?, ?, ?, ?)",
            (kind, content, cv.tobytes(), _stamp()))
    if verbose:
        print(f"  + add #{cur.lastrowid} [{kind}] {content}")
    return cur.lastrowid

def update_memory(mem_id, content, verbose=True):
    v = _vec(content)                                      # re-embed only this one row
    with connect() as con:
        con.execute("UPDATE memories SET content = ?, embedding = ?, created_at = ? WHERE id = ?",
                    (content, v.tobytes(), _stamp(), mem_id))
    if verbose:
        print(f"  ~ update #{mem_id} -> {content}")

def delete_memory(mem_id, verbose=True):
    with connect() as con:
        con.execute("DELETE FROM memories WHERE id = ?", (mem_id,))
    if verbose:
        print(f"  - delete #{mem_id}")

def clear_memories():
    with connect() as con:
        con.execute("DELETE FROM memories")

clear_memories()   # start clean for the demo
print("ready")

## Step 4 - Semantic recall: which memories are worth showing *this* turn

"Long-term" is a row's **lifetime** (on disk, every chat, until you delete it). "Semantic
recall" is **how a row is chosen** for one turn: embed the new message, compare it to every
memory's stored vector, keep the close ones.

- **preferences** -> long-term **and always applied** (handled in Step 6, not here)
- **facts / projects** -> long-term **but only recalled when relevant** - this function

Only the query is embedded here; the memory vectors were computed at write time (Step 3).

In [ ]:
def recall(query, k=4, threshold=0.15):
    rows = all_memories_vec()
    if not rows:
        return []
    q = _vec(query)                                        # one embed: the incoming message
    scored = sorted(((rid, rk, rc, float(q @ rv)) for rid, rk, rc, rv in rows),
                    key=lambda x: x[3], reverse=True)
    return [(rid, rk, rc) for rid, rk, rc, s in scored[:k] if s > threshold]

print("recall test (no memories yet):", recall("what should I cook?"))

## Step 5 - After each message, the model reconciles memory

One LLM call. It sees the **current memory list (with ids)** and the new message, and
returns a list of operations. Small talk and one-off questions produce `[]`.

In [ ]:
import json, re

RECONCILE_PROMPT = """You maintain a long-term memory about ONE user.

CURRENT MEMORIES:
{current}

The user just wrote:
"{msg}"

Decide how memory should change. Output ONLY a JSON list of operations:
  {{"op": "add", "kind": "preference|fact|project", "content": "..."}}
  {{"op": "update", "id": <existing id>, "content": "..."}}
  {{"op": "delete", "id": <existing id>}}

Rules:
- add    : a NEW durable fact / preference / project detail about the user
- update : the user changed something already stored (same topic, new value)
- delete : the user retracts or contradicts a stored memory
- ignore one-off questions and small talk -> return []"""

def _current_block():
    rows = all_memories()
    return "\n".join(f"#{i} [{k}] {c}" for i, k, c in rows) or "(none)"

def reconcile_memory(user_message, verbose=True):
    raw = llm.invoke(RECONCILE_PROMPT.format(current=_current_block(), msg=user_message))
    m = re.search(r"\[.*\]", raw, re.DOTALL)
    try:
        ops = json.loads(m.group(0)) if m else []
    except Exception:
        ops = []
    touched = []
    for op in ops:
        try:
            if op["op"] == "add":
                touched.append(add_memory(op.get("kind", "fact"), op["content"].strip(), verbose))
            elif op["op"] == "update":
                update_memory(int(op["id"]), op["content"].strip(), verbose); touched.append(int(op["id"]))
            elif op["op"] == "delete":
                delete_memory(int(op["id"]), verbose); touched.append(int(op["id"]))
        except Exception as e:
            if verbose:
                print("  (skipped a bad op:", e, ")")
    if verbose and not ops:
        print("  (no memory change)")
    return touched

print("chit-chat message:")
reconcile_memory("hey, what's up?")
print("\nsubstantive message:")
reconcile_memory("I'm vegetarian and I always want recipes as short numbered steps.")
print("\nmemory now:", all_memories())

## Step 6 - `chat_with_memory()` - apply preferences, recall facts, answer, reconcile

- **preferences** from memory are **always** put in the system prompt
- **facts / projects** are added only if `recall()` finds them relevant to this message
- after replying, `reconcile_memory()` updates the store

That is **two** LLM calls per turn: one to answer, one to reconcile. The Gradio app in
Step 12 shows the answer first (streamed) and only then reconciles, so the reconcile call
is off the critical path - you are reading the reply while memory updates in the background.

In [ ]:
def build_system_prompt(recalled):
    prefs = [c for i, k, c in all_memories() if k == "preference"]
    facts = [f"({k}) {c}" for i, k, c in recalled if k != "preference"]
    parts = ["You are a helpful assistant."]
    if prefs:
        parts.append("ALWAYS follow these user preferences:\n" + "\n".join(f"- {p}" for p in prefs))
    if facts:
        parts.append("Relevant things you know about this user:\n" + "\n".join(f"- {f}" for f in facts))
    return "\n\n".join(parts)

def chat_with_memory(user_text, history=None, verbose=True):
    history = history or []
    recalled = recall(user_text)
    system = build_system_prompt(recalled)
    if verbose:
        print("SYSTEM PROMPT:\n  " + system.replace("\n", "\n  "))

    messages = ([{"role": "system", "content": system}]
                + [{"role": r, "content": c} for r, c in history]
                + [{"role": "user", "content": user_text}])
    reply = llm.invoke(messages)

    if verbose:
        print("\nMEMORY RECONCILE:")
    touched = reconcile_memory(user_text, verbose=verbose)
    return reply, recalled, touched

print(chat_with_memory("What is RAG in one line?")[0])

## Step 7 - The real use case, part 1: no memory yet

Ask a question cold. The answer is generic - the assistant knows nothing about you or how
you like answers.

In [ ]:
clear_memories()

def plain_chat(text):
    return llm.invoke([{"role": "user", "content": text}])

print("--- PLAIN (no memory) ---")
print(plain_chat("What should I make for dinner tonight?")[:1200])

## Step 8 - Part 2: a few real messages -> the model fills memory

Each turn is 2 model calls (answer + reconcile), so we pause a beat between messages to
stay under the free-tier rate limit.

In [ ]:
import time

for msg in [
    "Hi! I'm vegetarian and I'm allergic to peanuts.",
    "I'm trying to meal-prep lunches for the week.",
    "Please give recipes as short numbered steps, no long intros.",
]:
    print(f"\nUSER: {msg}")
    chat_with_memory(msg, verbose=True)
    time.sleep(3)

print("\n=== memory now ===")
for i, k, c in all_memories():
    print(f"  #{i} [{k}] {c}")

## Step 9 - Part 3: brand-new conversation, same question

Fresh history (`history=[]`), so the assistant relies purely on long-term memory. Compare
this answer to Step 7's - it should now be **numbered steps**, **vegetarian**, and
**peanut-free**, none of which are in the question.

In [ ]:
answer, recalled, _ = chat_with_memory(
    "What should I make for dinner tonight?", history=[], verbose=True)
print("\n--- WITH MEMORY ---")
print(answer)

## Step 10 - Part 4: the plan changes -> the model *updates* a memory (no duplicate)

In [ ]:
chat_with_memory("Update: I've stopped meal-prepping, now I'm learning to bake sourdough.", verbose=True)
print("\n=== memory now ===")
for i, k, c in all_memories():
    print(f"  #{i} [{k}] {c}")

## Step 11 - Part 5: *you* delete a memory -> behaviour changes

In [ ]:
for i, k, c in all_memories():
    if k == "preference" and "step" in c.lower():
        delete_memory(i)

answer, _, _ = chat_with_memory("Give me a quick idea for a weeknight dinner.",
                                history=[], verbose=False)
print("--- after deleting the 'numbered steps' preference ---")
print(answer)

## Step 12 - Gradio app: chat + a memory panel you control

Chat on the left. On the right: every memory, plus what was **recalled** and **changed**
this turn, and buttons to delete one or clear all.

`ui_send` is a **generator**: it recalls (one embed), streams the answer token by token,
and only *after* the answer is complete does it run `reconcile_memory` and refresh the
panel. So the slow second LLM call happens while you are already reading.

In [ ]:
import gradio as gr

def memory_table():
    return [[i, k, c] for i, k, c in all_memories()]

def ui_send(user_text, history):
    history = history or []
    if not user_text.strip():
        yield history, "", memory_table(), "(none)", "(none)"
        return

    recalled = recall(user_text)
    system = build_system_prompt(recalled)
    msgs = ([{"role": "system", "content": system}]
            + [{"role": m["role"], "content": m["content"]} for m in history]
            + [{"role": "user", "content": user_text}])
    recalled_txt = "\n".join(f"({k}) {c}" for _, k, c in recalled) or "(none)"

    view = history + [{"role": "user", "content": user_text},
                      {"role": "assistant", "content": ""}]
    yield view, "", memory_table(), recalled_txt, "..."

    reply = ""
    for piece in llm.stream(msgs):                       # 1. stream the answer
        reply += piece
        view[-1]["content"] = reply
        yield view, "", memory_table(), recalled_txt, "..."

    touched = reconcile_memory(user_text, verbose=False)  # 2. then update memory
    changed_txt = ", ".join(f"#{i}" for i in touched) or "(none)"
    yield view, "", memory_table(), recalled_txt, changed_txt

def ui_delete(mem_id):
    if str(mem_id).strip():
        delete_memory(int(mem_id))
    return memory_table()

def ui_clear():
    clear_memories()
    return memory_table()

with gr.Blocks(title="Lab 4 - Memory") as demo:
    gr.Markdown("# Lab 4 - Assistant with editable long-term memory")
    with gr.Row():
        with gr.Column(scale=3):
            chatbox = gr.Chatbot(height=420)
            msg = gr.Textbox(label="Message", placeholder="Tell it about yourself, then ask something...")
        with gr.Column(scale=2):
            gr.Markdown("### Long-term memory")
            mem = gr.Dataframe(headers=["id", "kind", "content"], interactive=False, wrap=True)
            with gr.Row():
                del_id = gr.Textbox(label="id to delete", scale=1)
                del_btn = gr.Button("Delete", scale=1)
            clear_btn = gr.Button("Clear all memories", variant="stop")
            recalled_box = gr.Textbox(label="recalled this turn", lines=3)
            changed_box = gr.Textbox(label="memory changed this turn", lines=2)

    demo.load(memory_table, outputs=mem)
    msg.submit(ui_send, [msg, chatbox], [chatbox, msg, mem, recalled_box, changed_box])
    del_btn.click(ui_delete, del_id, mem)
    clear_btn.click(ui_clear, outputs=mem)

demo.launch(debug=False)

## Recap

- **Short-term** = the message list for the current chat (Lab 3). **Long-term** = a
  `memories` table that outlives every chat.
- **"Long-term" vs "semantic recall"** are not two stores - one table. Long-term is the
  *lifetime*; semantic recall is *how a row is picked for a turn* (embed the message,
  compare to each memory's vector, use the close ones). Preferences skip the filter - they
  always apply.
- The model **reconciles** memory (add / update / delete) after each turn; **you** can
  delete anything - nothing is kept against your wishes.
- **Speed**: each memory is embedded once at write time and the vector is stored, so a turn
  does exactly **one** embed (your message). The answer **streams**; the reconcile call
  runs after it, off the critical path.

### Exercises
1. Add an `importance` score and recall by `similarity * importance`.
2. Store the message each memory came from, and show it in the panel ("why do you know this?").
3. Give the Lab 3 chat store this memory table, so the ChatGPT clone remembers users too.
4. Print `time.time()` around `recall()` with 2 memories vs 50 - confirm it barely moves now
   that vectors are stored (before this fix it was linear in the table size).
